> 在线讲义：[第 4 章 · 工作区、colcon 和第一个包](https://wikieden.github.io/ros2-from-scratch/lessons/04-workspace-package.html) · 本书《从零学 ROS 2》

# 第 4 章 · 工作区、colcon 和第一个包

本章回答：自己写的 ROS 代码放哪？怎么编译？编译完为什么还要 source？"包"里必须有哪些文件？

配套讲义：仓库里 `ros2/notes/lessons/04-workspace-package.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：本机没有 ROS 2 和 colcon。这里把三条规则写成函数：source 之后按"上层先、底层后"找包；colcon 按 package.xml 的依赖排顺序、没依赖关系的并行；setup.py 的 console_scripts 一行变成一个可执行文件名。

In [ ]:
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 1 步 overlay | 同名包先找上层 | `find_package('demo_nodes_py', layers)` | 来自 `~/ros2_ws/install` |
| 第 1 步 underlay | 上层没有就去底层 | `find_package('rclpy', layers)` | 来自 `/opt/ros/lyrical` |
| 第 3 步 colcon | A 依赖 B 先编 B，互不依赖的并行 | `build_order(deps)` | 3 轮：[description, msgs] → [driver] → [bringup] |
| 第 2 步 entry_points | 命令名 = 模块路径:函数名 | `parse_entry_point(...)` | `my_node` / `my_package.my_node` / `main` |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**函数 `build_order`**：colcon 的排法：每一轮把"依赖都已编完"的包挑出来一起编（同一轮的可以并行）。

In [ ]:
def build_order(deps: dict[str, set[str]]) -> list[list[str]]:
    """colcon 的排法：每一轮把"依赖都已编完"的包挑出来一起编（同一轮的可以并行）。"""
    done: set[str] = set()
    rounds = []
    while len(done) < len(deps):
        ready = sorted(p for p, d in deps.items() if p not in done and d <= done)
        if not ready:
            raise ValueError("依赖有环，排不出顺序")
        rounds.append(ready)
        done |= set(ready)
    return rounds

**函数 `find_package`**：layers 按 source 后的优先级排好：最后 source 的工作区排最前。返回包是从哪一层找到的。

In [ ]:
def find_package(name: str, layers: list[tuple[str, set[str]]]) -> str | None:
    """layers 按 source 后的优先级排好：最后 source 的工作区排最前。返回包是从哪一层找到的。"""
    for layer_name, packages in layers:
        if name in packages:
            return layer_name
    return None

**函数 `parse_entry_point`**：'my_node = my_package.my_node:main' → 命令名、模块路径、函数名。

In [ ]:
def parse_entry_point(line: str) -> dict:
    """'my_node = my_package.my_node:main' → 命令名、模块路径、函数名。"""
    command, target = (s.strip() for s in line.split("="))
    module, func = target.split(":")
    return {"ros2 run 用的名字": command, "模块": module, "函数": func}

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 · overlay 在上、underlay 在下：同名包先找上层")
underlay = ("/opt/ros/lyrical", {"rclpy", "turtlesim", "demo_nodes_py"})
overlay = ("~/ros2_ws/install", {"my_package", "demo_nodes_py"})  # 假设你在上层改了一份 demo_nodes_py
layers = [overlay, underlay]  # 先 source 底层、再 source 上层 → 上层排在前面
for pkg in ["my_package", "demo_nodes_py", "rclpy"]:
    print(f"  {pkg:<14} 来自 {find_package(pkg, layers)}")

In [ ]:
print("\n第二步 · colcon 按依赖排顺序：A 依赖 B 就先编 B，互不依赖的同一轮并行")
deps = {
    "robot_msgs": set(),
    "robot_description": set(),
    "robot_driver": {"robot_msgs"},
    "robot_bringup": {"robot_driver", "robot_description"},
}  # 四个包的名字和依赖是编的
for i, rnd in enumerate(build_order(deps), 1):
    print(f"  第 {i} 轮: {rnd}")

In [ ]:
print("\n第三步 · setup.py 的 entry_points：一行登记一个可执行文件")
print(" ", parse_entry_point("my_node = my_package.my_node:main"))

## 3. 代码里才看得到的

1. **例子里的包名和依赖是编的**；`demo_nodes_py` 在两层都有，只为演示"同名先找上层"。
2. **`d <= done`** 是集合的"是不是子集"：一个包的依赖全在"已编完"里，它才能进这一轮。依赖有环时排不出顺序，函数会报错。
3. **colcon 只认 package.xml 里写的依赖**：代码里 import 了但没写进去的，它不知道，这正是"依赖要写全"的原因。

## 4. 自己改一改

- 在 `deps` 里加一个 `robot_tests`，依赖 `robot_bringup`，看多出第几轮；
- 把 `layers` 的顺序倒过来（相当于先 source 上层、再 source 底层），看 `demo_nodes_py` 从哪来。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **工作区**：源码进 src，产物三兄弟（build、install、log）。
- **叠层**：先 source 底层，再 source 上层；同名上层优先；新终端里 source。
- **包**：`package.xml` 是身份证，依赖写全；Python 入口写进 `setup.py`。
- **构建**：rosdep 装依赖，colcon 排顺序；Python 用 `--symlink-install`，改了不用重编。